# Association rules: co-occurring diagnostic features

This notebook walks through the rule mining in the order the coursework took
it. It calls the same functions as `analysis/m03_association_rules.py`, and
the write-up with every table and figure is
[docs/03-association-rules.md](../docs/03-association-rules.md).

The Wisconsin diagnostic breast cancer data (Wolberg et al. 1995) holds thirty
continuous features computed from images of fine needle aspirates and a
malignant or benign diagnosis for each of 569 samples. Apriori (Agrawal et al.
1993) is unsupervised; it receives 92 items with no distinguished one among
them and returns every frequent combination.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Every output directory is redirected to a temporary location before the
# pipeline modules are imported, so this notebook writes nothing into
# results/, figures/ or data/processed/. The committed tables are read from
# results/ directly where the notebook quotes them.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
_scratch = tempfile.mkdtemp()
for _name in ("ML_METHODS_RESULTS", "ML_METHODS_FIGURES", "ML_METHODS_PROCESSED"):
    os.environ[_name] = _scratch
sys.path.insert(0, str(ROOT))

import numpy
import pandas

from src import config, data, evaluate, splits

RESULTS = ROOT / "results"
pandas.set_option("display.width", 120)
pandas.set_option("display.max_columns", 20)


def recorded(prefix=""):
    """The committed metrics record, as a dictionary of strings."""
    table = pandas.read_csv(RESULTS / "metrics.csv", dtype=str)
    return {row.key: row.value for row in table.itertuples()
            if row.key.startswith(prefix)}

## Data

The measurements ship inside scikit-learn and are the same 569 by 30 table the
UCI record serves. `data.load_breast_cancer` names the diagnosis beside the
numeric code.

In [2]:
frame, counts = data.load_breast_cancer()
print(pandas.Series(counts))
frame.head()

rows_analyzed      569
features            30
rows_malignant     212
rows_benign        357
rows_incomplete      0
dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,malignant


## Inputs: from measurements to transactions

Apriori needs items. `m03.transactions` discretizes each feature into three
levels by one-dimensional k-means and one-hot encodes the result, then adds
the two diagnosis labels as items. The bin edges are returned beside the
table, because a rule naming level 0 of a feature means nothing without the
interval that level covers.

In [3]:
from analysis import m03_association_rules as m03

features = [column for column in frame.columns if column != "diagnosis"]
encoded, edges = m03.transactions(frame, features)
print(encoded.shape[0], "transactions,", encoded.shape[1], "items")
pandas.DataFrame(edges).head()

569 transactions, 92 items


,feature,edge_0,edge_1,edge_2,edge_3
0,mean radius,6.98100,12.895826,17.297098,28.1100
1,mean texture,9.71000,17.761656,23.384214,39.2800
2,mean perimeter,43.79000,84.659387,115.203946,188.5000
3,mean area,143.50000,775.522487,1436.765786,2501.0000
4,mean smoothness,0.05263,0.091549,0.109276,0.1634


## The support threshold decides which class can be found

An itemset cannot be more frequent than its rarest item. The malignant rate is
0.3726, so at the minimum support of 0.4 the coursework used, no frequent
itemset can contain the malignant diagnosis. The pipeline swept the threshold
downward to record where each class becomes reachable.

In [4]:
malignant_rate = counts["rows_malignant"] / counts["rows_analyzed"]
print("malignant rate", round(malignant_rate, 4), "| minimum support", config.MIN_SUPPORT)
pandas.read_csv(RESULTS / "m03_support_sweep.csv")

malignant rate 0.3726 | minimum support 0.4


,min_support,frequent_itemsets,itemsets_with_malignant,itemsets_with_benign
0,0.4,3653,0,792
1,0.3,9886,1,1735
2,0.2,27045,31,4104
3,0.1,80058,1699,8716


![Frequent itemsets naming each class against the support threshold.](../figures/fig08_rules_support_sweep.png)

## Training: mining at the coursework's threshold

Itemsets are bounded at length 4 and rules are kept at a conviction of at
least 10. The rules whose only consequent is the diagnosis are then selected
and ranked on a total order, so ties on lift do not reorder between runs.

In [5]:
from mlxtend.frequent_patterns import apriori, association_rules

itemsets = apriori(encoded, min_support=config.MIN_SUPPORT, use_colnames=True,
                   max_len=config.MAX_ITEMSET_LEN)
rules = association_rules(itemsets, num_itemsets=len(encoded),
                          metric="conviction", min_threshold=config.MIN_CONVICTION)
selected = m03.diagnosis_rules(rules)
print(len(itemsets), "frequent itemsets;", len(rules), "rules;",
      len(selected), "with the diagnosis as their only consequent")
selected["consequents"].map(lambda items: next(iter(items))).value_counts()

3653 frequent itemsets; 5972 rules; 321 with the diagnosis as their only consequent


consequents
diagnosis_benign    321
Name: count, dtype: int64

In [6]:
ranked = sorted(
    [{"antecedent": ", ".join(sorted(row.antecedents)),
      "consequent": next(iter(row.consequents)),
      "support": round(float(row.support), 4),
      "confidence": round(float(row.confidence), 4),
      "lift": round(float(row.lift), 4),
      "conviction": round(float(row.conviction), 1)}
     for row in selected.itertuples()],
    key=lambda rule: (-rule["lift"], -rule["support"], -rule["confidence"], rule["antecedent"]))
pandas.DataFrame(ranked).head(10)

,antecedent,consequent,support,confidence,lift,conviction
0,"mean concave points_0, radius error_0, worst p...",diagnosis_benign,0.4956,0.9965,1.5882,105.4
1,"mean concave points_0, perimeter error_0, wors...",diagnosis_benign,0.4938,0.9965,1.5882,105.1
2,"mean concave points_0, radius error_0, worst r...",diagnosis_benign,0.4798,0.9964,1.5880,102.1
3,"mean concave points_0, perimeter error_0, wors...",diagnosis_benign,0.4780,0.9963,1.5880,101.7
4,"mean concavity_0, perimeter error_0, worst per...",diagnosis_benign,0.4745,0.9963,1.5880,101.0
5,"mean concavity_0, radius error_0, worst perime...",diagnosis_benign,0.4745,0.9963,1.5880,101.0
6,"mean concavity_0, perimeter error_0, worst rad...",diagnosis_benign,0.4552,0.9962,1.5877,96.9
7,"mean concavity_0, radius error_0, worst radius_0",diagnosis_benign,0.4552,0.9962,1.5877,96.9
8,"radius error_0, worst compactness_0, worst per...",diagnosis_benign,0.4552,0.9962,1.5877,96.9
9,"perimeter error_0, worst compactness_0, worst ...",diagnosis_benign,0.4534,0.9961,1.5877,96.5


## Results

All 321 diagnosis rules name the benign class. The benign rate is 0.6274, so
the lift of such a rule cannot exceed 1.594, and the best rule reaches 1.588
at a confidence of 0.9965: of the 283 samples in the lowest level of mean
concave points, of radius error and of worst perimeter, 282 are benign.

### The feature selection reading

Every feature appearing in the antecedent of a diagnosis rule, with the number
of rules naming it.

In [7]:
appearances = {}
for antecedent in selected["antecedents"]:
    for item in antecedent:
        appearances[item] = appearances.get(item, 0) + 1
items = pandas.Series(appearances).sort_values(ascending=False)
print(len(items), "items from", len({item.rsplit("_", 1)[0] for item in items.index}), "features")
items

19 items from 19 features


worst perimeter_0            116
worst radius_0                95
worst concavity_0             74
worst concave points_0        68
worst area_0                  61
mean concave points_0         60
mean area_0                   58
mean concavity_0              53
radius error_0                52
perimeter error_0             52
area error_0                  51
worst compactness_0           39
mean perimeter_0              32
fractal dimension error_0     31
concavity error_0             30
mean radius_0                 15
mean compactness_0            13
compactness error_0            7
worst fractal dimension_0      1
dtype: int64

![The items appearing in the antecedents of the diagnosis rules.](../figures/fig09_rules_selected_items.png)

## Discussion

Every one of the 19 selected items is the lowest level of its feature, which
follows from the threshold: an item must hold in at least 40 percent of
samples to enter a frequent itemset, and for these features only the lowest
level is that common. The 19 are the size and boundary-shape measurements.
Radius, perimeter, area, concavity and compactness each appear in all three
released forms, and texture, smoothness and symmetry appear in no rule. The
mining step gave the diagnosis no special standing among the 92 items, so
this selection was reached by counting co-occurrences.

The rules describe the benign half of the feature space and nothing else.
Support and confidence are counts on all 569 samples with no held-out
partition, and rules that exchange radius for perimeter or area are not
independent findings, since those are three functions of one geometry. A
threshold near 0.1 would be needed before enough malignant itemsets exist to
rank, and the write-up records what the sweep found there.